### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="Gemma2-9b-It",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x11d48be90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1196ea000>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [4]:
result

AIMessage(content='Hello - Bonjour\n\nHow are you? - Comment allez-vous ? \n\n\nLet me know if you\'d like to know other ways to say "how are you?" \n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 21, 'total_tokens': 60, 'completion_time': 0.070909091, 'prompt_time': 0.002110129, 'queue_time': 0.08377702, 'total_time': 0.07301922}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--15caf4c8-0dd9-4b3b-9b35-f405051439b1-0', usage_metadata={'input_tokens': 21, 'output_tokens': 39, 'total_tokens': 60})

In [5]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Hello - Bonjour\n\nHow are you? - Comment allez-vous ? \n\n\nLet me know if you\'d like to know other ways to say "how are you?" \n'

In [6]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

"Here's the translation:\n\n* **Hello** - Bonjour\n* **How are you?** - Comment allez-vous ? (formal) or Comment vas-tu ? (informal) \n\n\nLet me know if you'd like me to translate anything else!\n"

In [7]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Trnaslate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [8]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [9]:
result.to_messages()

[SystemMessage(content='Trnaslate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [10]:
##Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour \n\n\n'